## Sumativa 02 – Análisis de datos de Juegos Olímpicos con Apache Spark

**MCDI502: Gestión de Datos y Tecnologías Big Data**  

**Proyecto:** Sumativa 02 – Análisis de datos de Juegos Olímpicos con Apache Spark  
**Dataset:** Juegos Olímpicos  
**Integrantes:** Enzo Pinilla, Claudio Alarcón y Luis Rodrigo Espinoza  
**Docente:** Eduardo Navarro Lorenzo


## 1. Configuración inicial del entorno

En esta sección se prepara el entorno necesario para utilizar Apache Spark.

Se verifica la instalación de Java, se instalan `pyspark` y `findspark` si no están disponibles, se crean las variables de entorno y se inicia la `SparkSession` junto con el `SparkContext`.

### 1.1. Instalar y verificar Apache Spark y Java

### 1.2. Crear las variables de entorno necesarias


### 1.3. Crear la SparkSession con el nombre `OlimpiadasAnalysis` y obtener el SparkContext

In [10]:
# Configuración del entorno de Apache Spark.
# PySpark y findspark se instalan automáticamente si no están disponibles.
# Java debe estar instalado en el equipo y se verifica antes de iniciar Spark.

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

def instalar_paquete_si_falta(nombre_paquete):
    if importlib.util.find_spec(nombre_paquete) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", nombre_paquete]
        )

instalar_paquete_si_falta("pyspark")
instalar_paquete_si_falta("findspark")

import pyspark
import findspark

# Detectar JAVA_HOME según el sistema operativo.
if sys.platform == "darwin":
    try:
        java_home = subprocess.check_output(
            ["/usr/libexec/java_home"]
        ).decode().strip()
    except subprocess.CalledProcessError as error:
        raise EnvironmentError(
            "Java no está instalado. En macOS puede instalarse con: "
            "brew install openjdk@17"
        ) from error
else:
    java_ejecutable = shutil.which("java")

    if java_ejecutable is None:
        raise EnvironmentError(
            "Java no está instalado o no se encuentra en el PATH."
        )

    java_real = Path(java_ejecutable).resolve()
    java_home = str(java_real.parent.parent)

os.environ["JAVA_HOME"] = java_home
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)

findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("OlimpiadasAnalysis")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

# Se utiliza la carpeta sumativa cuando existe.
# En caso contrario, se buscan los archivos junto al notebook.
data_dir = Path.cwd() / "sumativa"

if not data_dir.exists():
    data_dir = Path.cwd()

print("Versión de Java:")
subprocess.run(["java", "-version"], check=False)

print("\nJAVA_HOME =", os.environ["JAVA_HOME"])
print("SPARK_HOME =", os.environ["SPARK_HOME"])
print("SparkSession creada:", spark.conf.get("spark.app.name"))
print("SparkContext activo:", sc)
print("Directorio de datos:", data_dir.resolve())

print("\nArchivos disponibles:")
for archivo in sorted(data_dir.iterdir()):
    print("-", archivo.name)


Versión de Java:

JAVA_HOME = /Library/Java/JavaVirtualMachines/jdk-18.0.1.1.jdk/Contents/Home
SPARK_HOME = /Users/sauriomac/Documents/spark-semana2/.venv/lib/python3.10/site-packages/pyspark
SparkSession creada: OlimpiadasAnalysis
SparkContext activo: <SparkContext master=local[*] appName=OlimpiadasAnalysis>
Directorio de datos: /Users/sauriomac/Documents/spark-semana2/sumativa

Archivos disponibles:
- deportista.csv
- deportista2.csv
- equipo.csv
- evento.csv
- juegos.json
- resultados.csv


java version "18.0.1.1" 2022-04-22
Java(TM) SE Runtime Environment (build 18.0.1.1+2-6)
Java HotSpot(TM) 64-Bit Server VM (build 18.0.1.1+2-6, mixed mode, sharing)


## 2. RDDs: creación y unión (ID 3.1)



### 2.1 Creen un RDD llamado deportista con 6 particiones, leyendo el archivo deportista.csv.


In [11]:
# Se crea el RDD deportista desde deportista.csv con 6 particiones.

deportista = sc.textFile(
    str(data_dir / "deportista.csv"),
    minPartitions=6
)

print("Particiones del RDD deportista:", deportista.getNumPartitions())
print("Primer registro:", deportista.first())


Particiones del RDD deportista: 6
Primer registro: 1,A Dijiang,1,24,180,80,199


### 2.2 Creen un RDD llamado deportista2, leyendo el archivo deportista2.csv.


In [12]:
# Se crea el RDD deportista2 desde deportista2.csv.

deportista2 = sc.textFile(
    str(data_dir / "deportista2.csv")
)

print("Particiones del RDD deportista2:", deportista2.getNumPartitions())
print("Primer registro:", deportista2.first())


Particiones del RDD deportista2: 2
Primer registro: 67787,Lee BongJu,1,27,167,56,970


### 2.3 Creen un nuevo RDD llamado deportistaTotal que contenga la unión de los RDDs deportista y deportista2.


In [13]:
# Se crea deportistaTotal mediante la unión de ambos RDDs.

deportistaTotal = deportista.union(deportista2)

print("RDD deportistaTotal creado correctamente.")


RDD deportistaTotal creado correctamente.


### 2.4 Muestren la cantidad de registros contenidos en deportistaTotal.


In [14]:
# Se muestra la cantidad total de registros.
# El resultado esperado con los archivos entregados es 135571.

cantidad_deportistas = deportistaTotal.count()

print("Cantidad de registros en deportistaTotal:", cantidad_deportistas)


Cantidad de registros en deportistaTotal: 135571


### 2.5 Conviertan el RDD deportistaTotal en un DataFrame llamado deportista con las siguientes columnas: "deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id".

In [15]:
# Se convierte el RDD deportistaTotal en un DataFrame llamado deportista.
# Se definen tipos numéricos utilizables para las operaciones posteriores.
#
# Existe una línea de deportista.csv con una coma adicional al final.
# Por esta razón se toman solamente los primeros siete campos.

import csv

from pyspark.sql.types import (
    StructField,
    StructType,
    IntegerType,
    DoubleType,
    StringType,
)

def separar_deportista(linea):
    campos = next(csv.reader([linea]))
    return campos[:7]

filas_deportistas = (
    deportistaTotal
    .filter(lambda linea: linea is not None and linea.strip() != "")
    .map(separar_deportista)
    .filter(lambda fila: len(fila) == 7)
    .map(
        lambda fila: (
            int(fila[0]),
            fila[1],
            int(fila[2]),
            int(fila[3]),
            int(fila[4]),
            float(fila[5]),
            int(fila[6]),
        )
    )
)

esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("genero", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("altura", IntegerType(), True),
    StructField("peso", DoubleType(), True),
    StructField("equipo_id", IntegerType(), True),
])

deportista = spark.createDataFrame(
    filas_deportistas,
    schema=esquema_deportista
)

deportista.show(20, truncate=False)
deportista.printSchema()
print("Tipos de datos:", deportista.dtypes)


+-------------+------------------------------+------+----+------+-----+---------+
|deportista_id|nombre                        |genero|edad|altura|peso |equipo_id|
+-------------+------------------------------+------+----+------+-----+---------+
|1            |A Dijiang                     |1     |24  |180   |80.0 |199      |
|2            |A Lamusi                      |1     |23  |170   |60.0 |199      |
|3            |Gunnar Nielsen Aaby           |1     |24  |0     |0.0  |273      |
|4            |Edgar Lindenau Aabye          |1     |34  |0     |0.0  |278      |
|5            |Christine Jacoba Aaftink      |2     |21  |185   |82.0 |705      |
|6            |Per Knut Aaland               |1     |31  |188   |75.0 |1096     |
|7            |John Aalberg                  |1     |31  |183   |72.0 |1096     |
|8            |Cornelia Cor Aalten Strannood |2     |18  |168   |0.0  |705      |
|9            |Antti Sami Aalto              |1     |26  |186   |96.0 |350      |
|10           |E

## 3. RDDs: transformaciones (ID 3.2)

### 3.1 Creen un RDD llamado MayorEdad que permita filtrar deportistas mayores de edad.


In [16]:
# Se crea un RDD llamado MayorEdad con deportistas de 18 años o más.

MayorEdad = filas_deportistas.filter(
    lambda fila: fila[3] >= 18
)

print("Tipo de objeto:", type(MayorEdad))
print("Cantidad de deportistas mayores de edad:", MayorEdad.count())


Tipo de objeto: <class 'pyspark.rdd.PipelinedRDD'>
Cantidad de deportistas mayores de edad: 123244


In [17]:
# Se muestran registros del RDD MayorEdad.

for registro in MayorEdad.take(10):
    print(registro)


(1, 'A Dijiang', 1, 24, 180, 80.0, 199)
(2, 'A Lamusi', 1, 23, 170, 60.0, 199)
(3, 'Gunnar Nielsen Aaby', 1, 24, 0, 0.0, 273)
(4, 'Edgar Lindenau Aabye', 1, 34, 0, 0.0, 278)
(5, 'Christine Jacoba Aaftink', 2, 21, 185, 82.0, 705)
(6, 'Per Knut Aaland', 1, 31, 188, 75.0, 1096)
(7, 'John Aalberg', 1, 31, 183, 72.0, 1096)
(8, 'Cornelia Cor Aalten Strannood ', 2, 18, 168, 0.0, 705)
(9, 'Antti Sami Aalto', 1, 26, 186, 96.0, 350)
(10, 'Einar Ferdinand Einari Aalto', 1, 26, 0, 0.0, 350)


### 3.2 Creen un RDD llamado Deportistas_mujer que muestre solo los deportistas del género femenino (sexo=2).


In [18]:
# Se crea un RDD llamado Deportistas_mujer con género femenino (genero = 2).

Deportistas_mujer = filas_deportistas.filter(
    lambda fila: fila[2] == 2
)

print("Tipo de objeto:", type(Deportistas_mujer))
print("Cantidad de deportistas mujeres:", Deportistas_mujer.count())


Tipo de objeto: <class 'pyspark.rdd.PipelinedRDD'>
Cantidad de deportistas mujeres: 33981


In [19]:
# Se muestran registros del RDD Deportistas_mujer.

for registro in Deportistas_mujer.take(10):
    print(registro)


(5, 'Christine Jacoba Aaftink', 2, 21, 185, 82.0, 705)
(8, 'Cornelia Cor Aalten Strannood ', 2, 18, 168, 0.0, 705)
(13, 'Minna Maarit Aalto', 2, 30, 159, 55.5, 350)
(14, 'Pirjo Hannele Aalto Mattila ', 2, 32, 171, 65.0, 350)
(21, 'Ragnhild Margrethe Aamodt', 2, 27, 163, 0.0, 742)
(22, 'Andreea Aanei', 2, 22, 170, 125.0, 861)
(26, 'Agnes Erika Aanonsen Eyde ', 2, 17, 169, 65.0, 742)
(29, 'Willemien Aardenburg', 2, 22, 0, 0.0, 705)
(37, 'Ann Kristin Aarnes', 2, 23, 182, 64.0, 742)
(49, 'Moonika Aava', 2, 24, 168, 65.0, 331)


### 3.3 Conviertan a mayúsculas todas las palabras del RDD deportistaTotal.

In [20]:
# Se convierte a mayúsculas el contenido completo de deportistaTotal.
# Se utiliza take(10) para evidenciar el cambio sin llevar todo el RDD al driver.

deportistaTotal_mayusculas = deportistaTotal.map(
    lambda linea: linea.upper()
)

for registro in deportistaTotal_mayusculas.take(10):
    print(registro)


1,A DIJIANG,1,24,180,80,199
2,A LAMUSI,1,23,170,60,199
3,GUNNAR NIELSEN AABY,1,24,0,0,273
4,EDGAR LINDENAU AABYE,1,34,0,0,278
5,CHRISTINE JACOBA AAFTINK,2,21,185,82,705
6,PER KNUT AALAND,1,31,188,75,1096
7,JOHN AALBERG,1,31,183,72,1096
8,CORNELIA COR AALTEN STRANNOOD ,2,18,168,0,705
9,ANTTI SAMI AALTO,1,26,186,96,350
10,EINAR FERDINAND EINARI AALTO,1,26,0,0,350



## 4. DataFrames: creación e integración (ID 3.1)




### 4.1 Creen los DataFrames Evento, Resultado, Equipos y Juego a partir de los archivos eventos.csv, equipos.csv, resultado.csv y juego.json.


In [21]:
# Se crean los DataFrames Evento, Resultado, Equipos y Juego
# utilizando los nombres reales de los archivos entregados.

import csv

from pyspark.sql.functions import col, when
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

ruta_evento = str(data_dir / "evento.csv")
ruta_resultados = str(data_dir / "resultados.csv")
ruta_equipos = str(data_dir / "equipo.csv")
ruta_juegos = str(data_dir / "juegos.json")


# ------------------------------------------------------------------
# DataFrame Evento
# ------------------------------------------------------------------
# Algunas líneas de evento.csv están encerradas completamente entre comillas
# y contienen comillas dobles internas. Se normalizan para conservar las
# descripciones que incluyen comas.

evento_rdd = sc.textFile(ruta_evento)
encabezado_evento = evento_rdd.first()

def leer_linea_evento(linea):
    linea = linea.strip()

    if linea.startswith('"') and linea.endswith('"'):
        linea = linea[1:-1].replace('""', '"')

    return next(csv.reader([linea]))

def convertir_entero_nullable(valor):
    if valor in ("", "NA", "#N/A"):
        return None
    return int(valor)

datos_evento = (
    evento_rdd
    .filter(lambda linea: linea != encabezado_evento)
    .filter(lambda linea: linea is not None and linea.strip() != "")
    .map(leer_linea_evento)
    .filter(lambda fila: len(fila) == 3)
    .map(
        lambda fila: (
            int(fila[0]),
            fila[1],
            convertir_entero_nullable(fila[2]),
        )
    )
)

esquema_evento = StructType([
    StructField("cod_evento", IntegerType(), False),
    StructField("evento", StringType(), True),
    StructField("deporte_id", IntegerType(), True),
])

Df_Evento = spark.createDataFrame(
    datos_evento,
    schema=esquema_evento
)


# ------------------------------------------------------------------
# DataFrame Resultado
# ------------------------------------------------------------------
# resultados.csv utiliza punto y coma como separador.
# evento_id contiene algunos valores #N/A, que se convierten a null.

Df_Resultado = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .csv(ruta_resultados)
    .select(
        col("resultado_id").cast("int").alias("resultado_id"),
        col("medalla").cast("string").alias("medalla"),
        col("deportista_id").cast("int").alias("deportista_id"),
        col("juego_id").cast("int").alias("juego_id"),
        when(
            col("evento_id").isin("#N/A", "NA", "")
            | col("evento_id").isNull(),
            None
        )
        .otherwise(col("evento_id").cast("int"))
        .alias("evento_id")
    )
)


# ------------------------------------------------------------------
# DataFrame Equipos
# ------------------------------------------------------------------

Df_Equipos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv(ruta_equipos)
)


# ------------------------------------------------------------------
# DataFrame Juego
# ------------------------------------------------------------------

Df_Juego = (
    spark.read
    .option("multiline", True)
    .json(ruta_juegos)
)

# Se crean también las variables con los nombres exactos solicitados
# por la pauta. Todas contienen DataFrames.
Evento = Df_Evento
Resultado = Df_Resultado
Equipos = Df_Equipos
Juego = Df_Juego


# Evidencia de carga, cantidad de registros y esquema.

dataframes_entrada = {
    "Evento": Evento,
    "Resultado": Resultado,
    "Equipos": Equipos,
    "Juego": Juego,
}

for nombre_df, dataframe in dataframes_entrada.items():
    print("\n", "=" * 60)
    print(nombre_df, "->", dataframe.count(), "filas")
    dataframe.show(10, truncate=False)
    dataframe.printSchema()



Evento -> 765 filas
+----------+---------------------------------------------------+----------+
|cod_evento|evento                                             |deporte_id|
+----------+---------------------------------------------------+----------+
|1         |Basketball Men's Basketball                        |1         |
|2         |Judo Men's Extra-Lightweight                       |2         |
|3         |Football Men's Football                            |3         |
|4         |Tug-Of-War Men's Tug-Of-War                        |4         |
|5         |Speed Skating Women's 500 metres                   |5         |
|6         |Speed Skating Women's 1000 metres                  |5         |
|7         |Cross Country Skiing Men's 10 kilometres           |6         |
|8         |Cross Country Skiing Men's 50 kilometres           |6         |
|9         |Cross Country Skiing Men's 10/15 kilometres Pursuit|6         |
|10        |Cross Country Skiing Men's 4 x 10 kilometres Relay |6  

### 4.2 Realicen la optimización de los DataFrames creados.


In [22]:
# Se optimizan los DataFrames para acelerar su reutilización.
# cache() se utiliza en los DataFrames pequeños.
# persist() se utiliza en los DataFrames de mayor tamaño.

from pyspark import StorageLevel

Df_Evento.cache()
Df_Equipos.cache()
Df_Juego.cache()

deportista.persist(StorageLevel.MEMORY_AND_DISK)
Df_Resultado.persist(StorageLevel.MEMORY_AND_DISK)

# count() materializa la optimización y deja evidencia de que los
# DataFrames se encuentran operativos.

dataframes_optimizados = {
    "deportista": deportista,
    "Df_Evento": Df_Evento,
    "Df_Resultado": Df_Resultado,
    "Df_Equipos": Df_Equipos,
    "Df_Juego": Df_Juego,
}

for nombre_df, dataframe in dataframes_optimizados.items():
    print(
        nombre_df,
        "->",
        dataframe.count(),
        "filas | almacenamiento:",
        dataframe.storageLevel
    )


deportista -> 135571 filas | almacenamiento: Disk Memory Serialized 1x Replicated
Df_Evento -> 765 filas | almacenamiento: Disk Memory Deserialized 1x Replicated


Df_Resultado -> 271116 filas | almacenamiento: Disk Memory Serialized 1x Replicated
Df_Equipos -> 1184 filas | almacenamiento: Disk Memory Deserialized 1x Replicated
Df_Juego -> 51 filas | almacenamiento: Disk Memory Deserialized 1x Replicated


### 4.3 Crearen un nuevo DataFrame que contenga los datos de todos los DataFrames (deportista, evento, equipo, resultado y juego).

Relaciones utilizadas:

- `Df_Resultado.deportista_id = deportista.deportista_id`
- `deportista.equipo_id = Df_Equipos.id`
- `Df_Resultado.evento_id = Df_Evento.cod_evento`
- `Df_Resultado.juego_id = Df_Juego.juego_id`

Se utiliza `Df_Resultado` como DataFrame principal, ya que cada registro representa la participación de un deportista en un evento y juego.


In [23]:
# Se integran los cinco DataFrames mediante sus identificadores.

r = Df_Resultado.alias("r")
d = deportista.alias("d")
e = Df_Equipos.alias("e")
ev = Df_Evento.alias("ev")
j = Df_Juego.alias("j")

olimpiadas_completo = (
    r
    .join(
        d,
        r["deportista_id"] == d["deportista_id"],
        "left"
    )
    .join(
        e,
        d["equipo_id"] == e["id"],
        "left"
    )
    .join(
        ev,
        r["evento_id"] == ev["cod_evento"],
        "left"
    )
    .join(
        j,
        r["juego_id"] == j["juego_id"],
        "left"
    )
    .select(
        r["resultado_id"],
        r["medalla"],

        d["deportista_id"],
        d["nombre"],
        d["genero"],
        d["edad"],
        d["altura"],
        d["peso"],
        d["equipo_id"],

        e["equipo"],
        e["sigla"],

        ev["cod_evento"].alias("evento_id"),
        ev["evento"],
        ev["deporte_id"],

        j["juego_id"],
        j["ano"],
        j["temporada"],
        j["ciudad"],
    )
)

print("Cantidad de filas del DataFrame integrado:", olimpiadas_completo.count())
print("Resultado esperado: 271116 filas")
olimpiadas_completo.show(20, truncate=False)


Cantidad de filas del DataFrame integrado: 271116
Resultado esperado: 271116 filas
+------------+-------+-------------+------------------------+------+----+------+----+---------+--------------+-----+---------+---------------------------------------------------+----------+--------+-------------+---------+--------+
|resultado_id|medalla|deportista_id|nombre                  |genero|edad|altura|peso|equipo_id|equipo        |sigla|evento_id|evento                                             |deporte_id|juego_id|ano          |temporada|ciudad  |
+------------+-------+-------------+------------------------+------+----+------+----+---------+--------------+-----+---------+---------------------------------------------------+----------+--------+-------------+---------+--------+
|1           |NA     |1            |A Dijiang               |1     |24  |180   |80.0|199      |China         |CHN  |1        |Basketball Men's Basketball                        |1         |39      |1992 Verano  |1992     

### 5. Paralelismo (ID 3.4) Paralelicen el DataFrame resultante a 5 particiones.


In [24]:
# Se redistribuye el DataFrame integrado en 5 particiones.

olimpiadas_completo = olimpiadas_completo.repartition(5)

print(
    "Cantidad de particiones:",
    olimpiadas_completo.rdd.getNumPartitions()
)


Cantidad de particiones: 5


## 6. Inspección del DataFrame (ID 3.1)

Se muestra la cantidad de filas, los tipos de datos y el esquema del DataFrame integrado.


In [25]:
# Se inspecciona el DataFrame olimpiadas_completo.

print(
    "Cantidad de filas en olimpiadas_completo:",
    olimpiadas_completo.count()
)

print("\nTipos de datos:")
print(
    "\n".join(
        f"- {nombre_columna}: {tipo_dato}"
        for nombre_columna, tipo_dato in olimpiadas_completo.dtypes
    )
)

print("\nEsquema del DataFrame:")
olimpiadas_completo.printSchema()


Cantidad de filas en olimpiadas_completo: 271116

Tipos de datos:
- resultado_id: int
- medalla: string
- deportista_id: int
- nombre: string
- genero: int
- edad: int
- altura: int
- peso: double
- equipo_id: int
- equipo: string
- sigla: string
- evento_id: int
- evento: string
- deporte_id: int
- juego_id: bigint
- ano: string
- temporada: bigint
- ciudad: string

Esquema del DataFrame:
root
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: string (nullable = true)
 |-- evento_id: integer (nullable = true)
 |-- evento: string (nullable = true)
 |-- deporte_id: integer (nullable = true)
 |-- juego_id: long (nullable = true)
 |-- 

### 7. Columnas calculadas


### 7.1 Creen una columna calculada llamada IMC que determine el Índice de Masa Corporal de cada deportista.

In [26]:
# Se calcula el Índice de Masa Corporal:
# IMC = peso / (altura en metros)²
#
# Los valores 0 de peso o altura representan datos no informados.
# En esos casos el IMC queda como null para evitar una división por cero.
from pyspark.sql.functions import round as spark_round, when, col

olimpiadas_calculado = olimpiadas_completo.withColumn(
    "IMC",
    when(
        (col("altura") > 0) & (col("peso") > 0),
        spark_round(
            col("peso").cast("double")
            / ((col("altura").cast("double") / 100) ** 2),
            2
        )
    ).otherwise(None)
)

olimpiadas_calculado.select(
    "deportista_id",
    "nombre",
    "altura",
    "peso",
    "IMC"
).show(20, truncate=False)

+-------------+-----------------------------+------+-----+-----+
|deportista_id|nombre                       |altura|peso |IMC  |
+-------------+-----------------------------+------+-----+-----+
|68448        |Lei Lei Win                  |160   |50.0 |19.53|
|22025        |Jonathan Thomas Jon Cleveland|178   |73.0 |23.04|
|22025        |Jonathan Thomas Jon Cleveland|178   |73.0 |23.04|
|18830        |Norman Casmir                |0     |0.0  |NULL |
|13250        |Cline Bonnet Marchand        |170   |55.0 |19.03|
|73501        |Geza Magyar                  |190   |91.0 |25.21|
|46009        |Adrien Hardy                 |197   |87.0 |22.42|
|77611        |Jennifer McHugh              |160   |50.0 |19.53|
|57716        |Oleg Karpov                  |165   |65.0 |23.88|
|53476        |Dorian Lance James           |169   |87.0 |30.46|
|45468        |Han MyungHee                 |0     |0.0  |NULL |
|27505        |Isabelle Demongeot           |169   |59.0 |20.66|
|53798        |Inge Janss

### 7.2 Creen una columna calculada llamada Descripción_sexo que determine el sexo de los deportistas (sexo=1 es hombre, sexo=2 es mujer).


In [27]:
# Se crea la descripción del sexo a partir de la columna genero
# genero = 1: Hombre
# genero = 2: Mujer

olimpiadas_calculado = olimpiadas_calculado.withColumn(
    "Descripción_sexo",
    when(col("genero") == 1, "Hombre")
    .when(col("genero") == 2, "Mujer")
    .otherwise("Sin información")
)

olimpiadas_calculado.select(
    "deportista_id",
    "nombre",
    "genero",
    "Descripción_sexo"
).show(60, truncate=False)

+-------------+-------------------------------------------+------+----------------+
|deportista_id|nombre                                     |genero|Descripción_sexo|
+-------------+-------------------------------------------+------+----------------+
|19116        |Osvaldo Cavagnaro                          |1     |Hombre          |
|63255        |Atta Kouakou                               |1     |Hombre          |
|29651        |Dou Zhaobo                                 |1     |Hombre          |
|24051        |Ndia Vanda Sousa Eloy Cruz                 |2     |Mujer           |
|12351        |Alain Blondel                              |1     |Hombre          |
|71543        |Eduardo Jos Lorenzo Casasnova              |1     |Hombre          |
|5384         |George Artin Tajirian                      |1     |Hombre          |
|9712         |Duarte Manuel Pinto Coelho de Almeida Bello|1     |Hombre          |
|76408        |Pedro Csar Mathey Hoke                     |1     |Hombre    

## 8. Agregaciones requeridas (ID 3.3)



### 8.1 Creen un DataFrame que muestre la cantidad de medallas (oro, bronce, plata) obtenidas por cada Equipo.


In [28]:
# Se filtran solamente las medallas válidas.
# Después se utiliza pivot para crear una columna por cada tipo de medalla.

from pyspark.sql.functions import (
    avg,
    coalesce,
    count,
    desc,
    lit,
    max as spark_max,
    min as spark_min,
    sum as spark_sum,
)

medallas_equipo = (
    olimpiadas_calculado
    .filter(col("medalla").isin("Gold", "Silver", "Bronze"))
    .withColumn(
        "equipo",
        coalesce(col("equipo"), lit("Sin equipo registrado"))
    )
    .groupBy("equipo")
    .pivot("medalla", ["Gold", "Silver", "Bronze"])
    .agg(count("*"))
    .fillna(0)
    .withColumnRenamed("Gold", "Oro")
    .withColumnRenamed("Silver", "Plata")
    .withColumnRenamed("Bronze", "Bronce")
    .orderBy(
        desc("Oro"),
        desc("Plata"),
        desc("Bronce")
    )
)

medallas_equipo.show(100, truncate=False)

+---------------------------------------------------+----+-----+------+
|equipo                                             |Oro |Plata|Bronce|
+---------------------------------------------------+----+-----+------+
|United States                                      |2515|1546 |1249  |
|Soviet Union                                       |1110|766  |728   |
|Germany                                            |636 |603  |651   |
|Italy                                              |536 |507  |487   |
|Great Britain                                      |531 |591  |578   |
|France                                             |452 |524  |582   |
|Sweden                                             |450 |479  |510   |
|Hungary                                            |429 |329  |363   |
|Canada                                             |422 |407  |407   |
|East Germany                                       |409 |325  |283   |
|Australia                                          |343 |453  |

### 8.2 Creen un DataFrame que obtenga la suma, promedio, máximo y mínimo de la edad de los deportistas, agrupado por el tipo de medalla obtenida.


In [29]:
# Las edades con valor 0 se excluyen porque representan datos no informados.

edad_por_medalla = (
    olimpiadas_calculado
    .filter(
        col("medalla").isin("Gold", "Silver", "Bronze")
        & (col("edad") > 0)
    )
    .groupBy("medalla")
    .agg(
        spark_sum("edad").alias("suma_edad"),
        spark_round(avg("edad"), 2).alias("promedio_edad"),
        spark_max("edad").alias("edad_maxima"),
        spark_min("edad").alias("edad_minima"),
    )
    .orderBy("medalla")
)

edad_por_medalla.show(truncate=False)

+-------+---------+-------------+-----------+-----------+
|medalla|suma_edad|promedio_edad|edad_maxima|edad_minima|
+-------+---------+-------------+-----------+-----------+
|Bronze |308071   |23.69        |72         |10         |
|Gold   |313374   |23.7         |64         |11         |
|Silver |304518   |23.75        |73         |11         |
+-------+---------+-------------+-----------+-----------+



### 8.3 Creen un nuevo DataFrame llamado temporada que obtenga la suma, promedio, máximo y mínimo de las alturas de los deportistas, agrupado por temporada.


In [30]:
# Las alturas con valor 0 se excluyen porque representan datos no informados.

temporada = (
    olimpiadas_calculado
    .filter(
        col("temporada").isNotNull()
        & (col("altura") > 0)
    )
    .groupBy("temporada")
    .agg(
        spark_sum("altura").alias("suma_altura"),
        spark_round(avg("altura"), 2).alias("promedio_altura"),
        spark_max("altura").alias("altura_maxima"),
        spark_min("altura").alias("altura_minima"),
    )
    .orderBy("temporada")
)

temporada.show(100, truncate=False)

+---------+-----------+---------------+-------------+-------------+
|temporada|suma_altura|promedio_altura|altura_maxima|altura_minima|
+---------+-----------+---------------+-------------+-------------+
|1896     |7946       |172.74         |188          |154          |
|1900     |20490      |176.64         |191          |153          |
|1904     |37443      |175.79         |195          |155          |
|1906     |45799      |178.21         |196          |165          |
|1908     |84333      |177.54         |201          |157          |
|1912     |127940     |177.45         |200          |157          |
|1920     |134802     |175.75         |197          |142          |
|1924     |170414     |174.96         |200          |142          |
|1928     |170783     |175.16         |211          |147          |
|1932     |211329     |174.22         |200          |147          |
|1936     |209463     |175.72         |205          |147          |
|1948     |205946     |176.17         |213      

### 8.4 Creen un nuevo DataFrame llamado sexo que obtenga la suma, promedio, máximo y mínimo de la edad de los deportistas, agrupado por sexo.

In [31]:
# Se agrupa por la columna calculada Descripción_sexo.
# Las edades con valor 0 se excluyen porque representan datos no informados.

sexo = (
    olimpiadas_calculado
    .filter(col("edad") > 0)
    .groupBy("Descripción_sexo")
    .agg(
        spark_sum("edad").alias("suma_edad"),
        spark_round(avg("edad"), 2).alias("promedio_edad"),
        spark_max("edad").alias("edad_maxima"),
        spark_min("edad").alias("edad_minima"),
    )
    .orderBy("Descripción_sexo")
)

sexo.show(truncate=False)

+----------------+---------+-------------+-----------+-----------+
|Descripción_sexo|suma_edad|promedio_edad|edad_maxima|edad_minima|
+----------------+---------+-------------+-----------+-----------+
|Hombre          |4581905  |24.43        |97         |10         |
|Mujer           |1619623  |21.86        |74         |11         |
+----------------+---------+-------------+-----------+-----------+



## 9. Verificación final de la rúbrica

Esta celda resume los productos solicitados y permite comprobar que el notebook se ejecutó completamente.


In [32]:
print("Verificación final")
print("- deportistaTotal:", cantidad_deportistas, "registros")
print("- deportista es DataFrame:", type(deportista))
print("- MayorEdad es RDD:", type(MayorEdad))
print("- Deportistas_mujer es RDD:", type(Deportistas_mujer))
print("- DataFrame integrado:", olimpiadas_completo.count(), "filas")
print("- Particiones:", olimpiadas_completo.rdd.getNumPartitions())
print("- Columnas calculadas:", "IMC" in olimpiadas_calculado.columns, "Descripción_sexo" in olimpiadas_calculado.columns)
print("- Agregaciones creadas: medallas_equipo, edad_por_medalla, temporada y sexo")


Verificación final
- deportistaTotal: 135571 registros
- deportista es DataFrame: <class 'pyspark.sql.dataframe.DataFrame'>
- MayorEdad es RDD: <class 'pyspark.rdd.PipelinedRDD'>
- Deportistas_mujer es RDD: <class 'pyspark.rdd.PipelinedRDD'>
- DataFrame integrado: 271116 filas
- Particiones: 5
- Columnas calculadas: True True
- Agregaciones creadas: medallas_equipo, edad_por_medalla, temporada y sexo


# Bibliografía


- Maldonado, Sebastián (2022), Analytics y big data: ciencia de los datos aplicada al mundo de los negocios.
https://unab.primo.exlibrisgroup.com/permalink/56UAB_INST/1ebbirc/cdi_askewsholts_vlebooks_9788418982637Links.

- López Fandiño, V. M. (2023). Sistemas de Big Data. Alfaomega Grupo Editor.
https://unab.primo.exlibrisgroup.com/permalink/56UAB_INST/1ebbirc/cdi_elibro_books_ELB235054Links.

